In [0]:
from LDCDataAccessLayerPy import KeyVaultManager, SharePointManager, SqlManager, databricks_init
from datetime import datetime, timedelta
from LDCDataAccessLayerPy import databricks_init, DataLakeManagerGen2
from io import BytesIO
import LDCDataAccessLayerPy
#Initiate the secret to access KeyVault secrets
databricks_init(dbutils, 'GO')
sp_mgr = SharePointManager()
sql_mgr = SqlManager()

import logging
logger = spark._jvm.org.apache.log4j
logging.getLogger("py4j").setLevel(logging.ERROR)

import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.colors import ListedColormap
from sklearn.cluster import KMeans
from openpyxl import load_workbook

from datetime import datetime, timedelta
import re
from dateutil.relativedelta import relativedelta
from LDCDataAccessLayerPy import PriceManager, graph
from LDCDataAccessLayerPy import ZemaManager
import math
import plotly.express as px

from datetime import datetime, timedelta
zema = ZemaManager()


In [0]:
end_date = datetime(2030, 1, 1)
url = "https://ldcom365.sharepoint.com"

In [0]:


today = datetime.today()

if today.weekday() == 0:  # Monday
    start_date = today - timedelta(days=3)
elif today.weekday() == 6:  # Sunday
    start_date = today - timedelta(days=2)
else:
    start_date = today - timedelta(days=1)

# Keep shifting start_date back until dataframe is not empty
while True:
    arg_sa = zema.get_curve(curve="P-FREIGHT-LDC-INPUT-PANAMAX-CORN-AR, Upriver+Bahia Blanca-SA, Dammam-USD-MT",period=f"{start_date}::{end_date}")
    arg_viet=zema.get_curve(curve="P-FREIGHT-LDC-INPUT-PANAMAX-CORN-AR, Upriver+Bahia Blanca-VN, Cai Mep-USD-MT", period=f"{start_date}::{end_date}")
    arg_kor=zema.get_curve(curve="P-FREIGHT-LDC-INPUT-PANAMAX-CORN-AR, Upriver+Bahia Blanca-KR, Incheon-USD-MT", period=f"{start_date}::{end_date}")
    arg_egy=zema.get_curve(curve="P-FREIGHT-LDC-INPUT-PANAMAX-CORN-AR, Upriver+Bahia Blanca-EG, El Dekheila-USD-MT", period=f"{start_date}::{end_date}")

    usg_sa=zema.get_curve(curve="P-FREIGHT-LDC-INPUT-PANAMAX-CORN-US-Port Allen-13.71m fw-SA-Yanbu-via Suez-V000008846-USD-MT", period=f"{start_date}::{end_date}")
    pnw_viet=zema.get_curve(curve="P-FREIGHT-LDC-INPUT-PANAMAX-CORN-US, Seattle-VN, Cai Mep-USD-MT", period=f"{start_date}::{end_date}")
    pnw_kor=zema.get_curve(curve="P-FREIGHT-LDC-INPUT-PANAMAX-CORN-US, Seattle-KR, Incheon-USD-MT", period=f"{start_date}::{end_date}")
    usg_egy=zema.get_curve(curve="P-FREIGHT-LDC-INPUT-PANAMAX-CORN-US, Port Allen-EG, El Dekheila-via Suez-USD-MT", period=f"{start_date}::{end_date}")
    usg_viet=zema.get_curve(curve="P-FREIGHT-LDC-INPUT-PANAMAX-CORN-US, Port Allen-VN, Cai Mep-via Suez-USD-MT", period=f"{start_date}::{end_date}")
    usg_kor=zema.get_curve(curve="P-FREIGHT-LDC-INPUT-PANAMAX-CORN-US, Port Allen-KR, Incheon-via Suez-USD-MT", period=f"{start_date}::{end_date}")

    bra_sa=zema.get_curve(curve="P-FREIGHT-LDC-INPUT-PANAMAX-CORN-BR, Santos-SA, Dammam-USD-MT", period=f"{start_date}::{end_date}")
    bra_viet=zema.get_curve(curve="P-FREIGHT-LDC-INPUT-PANAMAX-CORN-BR, Santos-VN, Cai Mep-USD-MT", period=f"{start_date}::{end_date}")
    bra_kor=zema.get_curve(curve="P-FREIGHT-LDC-INPUT-PANAMAX-CORN-BR, Santos-KR, Incheon-USD-MT", period=f"{start_date}::{end_date}")
    bra_egy=zema.get_curve(curve="P-FREIGHT-LDC-INPUT-PANAMAX-CORN-BR, Santos-EG, El Dekheila-USD-MT", period=f"{start_date}::{end_date}")

    ua_sa=zema.get_curve(curve="P-FREIGHT-LDC-INPUT-PANAMAX-CORN-UA-Odessa-SA-Dammam-via Suez-V000003503-USD-MT", period=f"{start_date}::{end_date}")
    ua_viet=zema.get_curve(curve="P-FREIGHT-LDC-INPUT-PANAMAX-CORN-UA, ODESSA-12.8M BW-VN, CAI MEP-VIA SUEZ-V000011177-USD-MT", period=f"{start_date}::{end_date}")
    ua_kor=zema.get_curve(curve="P-FREIGHT-LDC-INPUT-PANAMAX-CORN-UA, Odessa-KR, Incheon-USD-MT", period=f"{start_date}::{end_date}")
    ua_egy=zema.get_curve(curve="P-FREIGHT-LDC-INPUT-PANAMAX-CORN-UA, Odessa-EG, El Dekheila-USD-MT", period=f"{start_date}::{end_date}")

    if not arg_sa.empty:
        break
    else:
        start_date -= timedelta(days=1)


In [0]:
dfs=[arg_sa,arg_viet,arg_kor,arg_egy,usg_sa,pnw_viet,pnw_kor,usg_egy,usg_viet,usg_kor,bra_sa,bra_viet,bra_kor,bra_egy,ua_sa,ua_viet,ua_kor,ua_egy]
# Define corresponding names for clarity in outputs
df_names = ['arg_sa', 'arg_viet', 'arg_kor','arg_egy', 'usg_sa','pnw_viet', 'pnw_kor','usg_egy','usg_viet','usg_kor','bra_sa', 'bra_viet', 'bra_kor','bra_egy','ua_sa', 'ua_viet', 'ua_kor','ua_egy']

# Function to process each DataFrame
def process_df(df):
    df = df[df['observation'] == 'Last']
    df = df[['date', 'value', 'contract_year', 'contract_month']]

    df['contract_date'] = pd.to_datetime(
        dict(year=df['contract_year'], month=df['contract_month'], day=1)
    )

    return df

# Dictionary to store results
spot_dfs = {}

# Loop through your list
for df, name in zip(dfs, df_names):
    spot_dfs[f'spot_{name}'] = process_df(df)


spot_arg_sa = spot_dfs['spot_arg_sa']
spot_arg_viet = spot_dfs['spot_arg_viet']
spot_arg_kor = spot_dfs['spot_arg_kor']
spot_arg_egy = spot_dfs['spot_arg_egy']

spot_pnw_viet = spot_dfs['spot_pnw_viet']
spot_pnw_kor = spot_dfs['spot_pnw_kor']
spot_usg_sa = spot_dfs['spot_usg_sa']
spot_usg_egy = spot_dfs['spot_usg_egy']
spot_usg_viet = spot_dfs['spot_usg_viet']
spot_usg_kor = spot_dfs['spot_usg_kor']

spot_bra_sa = spot_dfs['spot_bra_sa']
spot_bra_viet = spot_dfs['spot_bra_viet']
spot_bra_kor = spot_dfs['spot_bra_kor']
spot_bra_egy = spot_dfs['spot_bra_egy']

spot_ua_sa = spot_dfs['spot_ua_sa']
spot_ua_viet = spot_dfs['spot_ua_viet']
spot_ua_kor = spot_dfs['spot_ua_kor']
spot_ua_egy = spot_dfs['spot_ua_egy']


In [0]:
import plotly.graph_objects as go

# Build lists as (name, dataframe)
df_sa = [("UPBB", spot_arg_sa), ("USG (to Yanbu)", spot_usg_sa), ("Santos", spot_bra_sa), ("Odessa", spot_ua_sa)]
df_kor = [("UPBB", spot_arg_kor), ("PNW", spot_pnw_kor),("USG (Suez)", spot_usg_kor), ("Santos", spot_bra_kor), ("Odessa", spot_ua_kor)]
df_egy = [("UPBB", spot_arg_egy), ("USG", spot_usg_egy), ("Santos", spot_bra_egy), ("Odessa", spot_ua_egy)]
df_viet = [("UPBB", spot_arg_viet), ("PNW", spot_pnw_viet),("USG (Suez)", spot_usg_viet), ("Santos", spot_bra_viet), ("Odessa", spot_ua_viet)]

def plot_df_list(named_df_list, title):
    fig = go.Figure()
    for name, df in named_df_list:
        fig.add_trace(
            go.Scatter(
                x=df["contract_date"],
                y=df["value"],
                mode="lines+markers",
                name=name.upper()  # trace name from tuple
            )
        )
    fig.update_layout(
        title=title,
        xaxis_title="Contract Date",
        yaxis_title="Freight",
        template="plotly_white",
        hovermode="x unified",
        height=700,
        width=1800,
    )
    return fig

# Build 4 figures
fig_sa = plot_df_list(df_sa, "Saudi Corn Freight rates (Dammam)")
fig_kor = plot_df_list(df_kor, "Korea Corn Freight rates (Incheon)")
fig_egy = plot_df_list(df_egy, "Egypt Corn Freight rates (El Dekheila)")
fig_viet = plot_df_list(df_viet, "Vietnam Corn Freight rates (Cai Mep)")

# Show them
fig_sa.show()
fig_kor.show()
fig_egy.show()
fig_viet.show()

fig_sa_html = fig_sa.to_html(include_plotlyjs='cdn', full_html=True)
fig_kor_html = fig_kor.to_html(include_plotlyjs='cdn', full_html=True)
fig_egy_html = fig_egy.to_html(include_plotlyjs='cdn', full_html=True)
fig_viet_html = fig_viet.to_html(include_plotlyjs='cdn', full_html=True)



In [0]:
# List to store pivoted DataFrames
pivoted_list = []

for df, name in zip(dfs, df_names):
    parts = name.split('_')
    origin = parts[0].upper()
    destination = parts[1].upper()
    
    df['origin'] = origin
    df['destination'] = destination
    df['month_year'] = df['contract_month'].astype(str) + '-' + df['contract_year'].astype(str)
    
    pivot_df = df.pivot_table(
        index=['destination','origin'],
        columns='month_year',
        values='value',
        aggfunc='first'
    ).reset_index()
    
    # Sort month-year columns chronologically
    month_cols = [c for c in pivot_df.columns if c not in ['destination','origin']]
    month_cols_sorted = sorted(
        month_cols,
        key=lambda x: pd.to_datetime(x, format='%m-%Y')
    )
    
    pivot_df = pivot_df[['destination','origin'] + month_cols_sorted]
    pivoted_list.append(pivot_df)

# Concatenate all into a single DataFrame
final_df = pd.concat(pivoted_list, ignore_index=True)


In [0]:
# Mapping dictionary
destination_mapping = {
    'SA': 'Saudi Arabia',
    'VIET': 'Vietnam',
    'KOR': 'Korea',
    'EGY': 'Egypt'
}

# Apply mapping to the destination column
final_df['destination'] = final_df['destination'].map(destination_mapping)
final_df.reset_index(drop=True)
final_df = final_df.sort_values(by=['destination','origin'])

excel_buffer_final_df = BytesIO()
final_df.to_excel(excel_buffer_final_df, index=False, sheet_name='Rates')
excel_buffer_final_df.seek(0)  # Reset the buffer's position



In [0]:
html_content = f"""
<!DOCTYPE html>
<html>
<head>
    <title>CORN PANAMAX FREIGHT RATES</title>
    <style>
        body {{
            font-family: Arial, sans-serif;
            margin: 20px;
        }}
        h1, h2 {{
            margin-top: 40px;
            color: #333;
        }}
        img {{
            width: 1800px;
            max-width: 100%;
            min-width: 1000px;
            display: block;
            margin-bottom: 30px;
        }}
    </style>
</head>
<body>
    <h1>Corn Freight rates</h1>
    <h2>To Korea (Incheon)</h2>
    <img src="fig_kor" alt="Panamax">
    <h2>To Vietnam (Cai Mep)</h2>
    <img src="fig_viet" alt="Panamax">
    <h2>To Egypt (El Dekheila)</h2>
    <img src="fig_egy" alt="Panamax">
    <h2>To Saudi Arabia (Dammam)</h2>
    <img src="fig_sa" alt="Panamax">

</body>
</html>
"""

In [0]:
corn_freight  = f"""
<!DOCTYPE html>
<html>
<head>
    <title>Corn CFR to destinations</title>
    <style>
        body {{
            font-family: Arial, sans-serif;
            padding: 20px;
        }}
        h2 {{
            margin-top: 40px;
            color: #2c3e50;
        }}
        .chart-container {{
            margin-bottom: 50px;
        }}
    </style>
</head>
<body>
    <h1>Corn Freight Rates</h1>
    <h2>Korea</h2>
    <div class="chart-container">{fig_kor_html}</div>
        <h2>Vietnam</h2>
    <div class="chart-container">{fig_viet_html}</div>
        <h2>Egypt</h2>
    <div class="chart-container">{fig_egy_html}</div>
        <h2>Saudi</h2>
    <div class="chart-container">{fig_sa_html}</div>
</body>
</html>
"""
corn_freight_report_bytes = corn_freight.encode("utf-8")

grains=['florian.girardi-ext@ldc.com','Roman.Avramishin@LDC.com','juan.garciafuentes@ldc.com','gonzalo.lascombes@ldc.com','juan.carnemolla@LDC.com','valentin.chiesa@ldc.com']
test_2=['florian.girardi-ext@ldc.com']
      
# Send email with embedded chart and table
LDCDataAccessLayerPy.mail.mail_send(
    to=grains,
    subject=f'CORN FREIGHT RATES as of {start_date.strftime("%d-%m")}',
    from_addr="florian.girardi-ext@ldc.com",
    mime_type="html",
    body=html_content, 
    html_images={"fig_sa":fig_sa,"fig_kor":fig_kor,"fig_egy":fig_egy,"fig_viet":fig_viet},
    attachment={"corn_freight.html": corn_freight_report_bytes,"freight_rates_corn.xlsx": excel_buffer_final_df}
)




# HISTORICAL FREIGHT

In [0]:
from dateutil.relativedelta import relativedelta

today = datetime.today()-relativedelta(months=6)

if today.weekday() == 0:  # Monday
    start_date = today - timedelta(days=3)
elif today.weekday() == 6:  # Sunday
    start_date = today - timedelta(days=2)
else:
    start_date = today - timedelta(days=1)

today

In [0]:
# === Step 1: Get curves ===
arg_sa = zema.get_curve(
    curve="P-FREIGHT-LDC-INPUT-PANAMAX-CORN-AR, Upriver+Bahia Blanca-SA, Dammam-USD-MT",
    period=f"{start_date}::{end_date}"
)
arg_viet = zema.get_curve(
    curve="P-FREIGHT-LDC-INPUT-PANAMAX-CORN-AR, Upriver+Bahia Blanca-VN, Cai Mep-USD-MT",
    period=f"{start_date}::{end_date}"
)
arg_kor = zema.get_curve(
    curve="P-FREIGHT-LDC-INPUT-PANAMAX-CORN-AR, Upriver+Bahia Blanca-KR, Incheon-USD-MT",
    period=f"{start_date}::{end_date}"
)
arg_egy = zema.get_curve(
    curve="P-FREIGHT-LDC-INPUT-PANAMAX-CORN-AR, Upriver+Bahia Blanca-EG, El Dekheila-USD-MT",
    period=f"{start_date}::{end_date}"
)

dfs_arg = [arg_sa, arg_viet, arg_kor, arg_egy]
df_names_arg = ["arg_sa", "arg_viet", "arg_kor", "arg_egy"]

# === Step 2: Process function ===
def process_df_arg(df, destination):
    df = df[df["observation"] == "Last"]
    df["month_year"] = df["contract_month"].astype(str) + '-' + df["contract_year"].astype(str)
    df = df[["date", "value", "month_year"]]
    df["Vessel"] = "PANAMAX"
    df["Origin"] = "UPBB"
    df["Destination"] = destination
    return df

# === Step 3: Build processed DataFrames ===
hist_arg_sa   = process_df_arg(arg_sa, "DAMMAM (SA)")
hist_arg_viet = process_df_arg(arg_viet, "CAI MEP (VIETNAM)")
hist_arg_kor  = process_df_arg(arg_kor, "INCHEON (KOREA)")
hist_arg_egy  = process_df_arg(arg_egy, "EL DEKHEILA (EGYPT)")

hist_arg_df = pd.concat([hist_arg_sa, hist_arg_viet, hist_arg_kor, hist_arg_egy])

# === Step 4: Pivot ===
pivot_df_arg = hist_arg_df.pivot_table(
    index=['date', 'Origin', 'Destination', 'Vessel'],
    columns='month_year',
    values='value',
    aggfunc='first'
).reset_index()

# === Step 5: Sort contract columns ===
fixed_cols_arg = ["date", "Origin", "Destination", "Vessel"]
contract_cols_arg = [c for c in pivot_df_arg.columns if c not in fixed_cols_arg]

def parse_contract(c):
    try:
        m, y = c.split("-")
        return pd.to_datetime(f"{y}-{m}-01")
    except:
        return pd.NaT

sorted_contracts_arg = sorted(contract_cols_arg, key=parse_contract)
pivot_df_arg = pivot_df_arg[fixed_cols_arg + sorted_contracts_arg]

# === Step 6: Export to Excel buffer ===
excel_buffer_pivot_df_arg = BytesIO()
pivot_df_arg.to_excel(excel_buffer_pivot_df_arg, index=False, sheet_name='Rates')
excel_buffer_pivot_df_arg.seek(0)


In [0]:
# Melt to long format
df_long_arg = pivot_df_arg.melt(
    id_vars=["date", "Destination", "Vessel", "Origin"],
    var_name="contract",
    value_name="freight_rate"
)

# Convert date
df_long_arg["date"] = pd.to_datetime(df_long_arg["date"], errors="coerce")

# Parse contracts into datetime for sorting
df_long_arg["contract_date"] = pd.to_datetime(df_long_arg["contract"], format="%m-%Y", errors="coerce")

# Keep valid values
df_long_arg = df_long_arg.dropna(subset=["freight_rate", "contract_date"])

# Color palette for contracts
palette = px.colors.qualitative.Set2 + px.colors.qualitative.Set1 + px.colors.qualitative.Pastel

# Loop over destinations (instead of origins)
figs_dest = {}
for dest in df_long_arg["Destination"].unique():
    fig = go.Figure()

    # Contracts sorted in chronological order
    contracts_sorted = sorted(df_long_arg["contract"].unique(),
                              key=lambda x: pd.to_datetime(x, format="%m-%Y"))
    color_map = {c: palette[i % len(palette)] for i, c in enumerate(contracts_sorted)}

    # Filter only this destination (always PANAMAX)
    df_sub = df_long_arg[df_long_arg["Destination"] == dest]

    for contract in contracts_sorted:
        df_c = df_sub[df_sub["contract"] == contract]
        if df_c.empty:
            continue
        fig.add_trace(go.Scatter(
            x=df_c["date"],
            y=df_c["freight_rate"],
            mode="lines",
            name=f"{contract}",
            line=dict(color=color_map[contract])
        ))

    # Add buttons to toggle contracts
    buttons = []
    for contract in contracts_sorted:
        visible = []
        for trace in fig.data:
            # show only traces of this contract
            if contract in trace.name:
                visible.append(True)
            else:
                visible.append(False)
        buttons.append(dict(
            label=contract,
            method="update",
            args=[{"visible": visible}]
        ))

    # Add ALL button to reset view
    buttons.insert(0, dict(
        label="ALL",
        method="update",
        args=[{"visible": [True] * len(fig.data)}]
    ))

    fig.update_layout(
        title=f"Freight Rate Evolution - UPBB to {dest}",
        xaxis_title="Date",
        yaxis_title="Freight Rate (USD/MT)",
        hovermode="x unified",
        updatemenus=[dict(
            type="buttons",
            direction="left",
            x=0,
            y=1.15,
            xanchor="left",
            yanchor="top",
            buttons=buttons,
            showactive=True
        )]
    )

    figs_dest[dest] = fig

# Display each chart
for dest, fig in figs_dest.items():
    fig.show()
    print(dest)



In [0]:
figs_html = {}
for dest, fig in figs_dest.items():
    figs_html[dest] = fig.to_html(include_plotlyjs="cdn", full_html=True)


# Example: access the HTML for MONTEVIDEO
fig_viet_html = figs_html.get("CAI MEP (VIETNAM)")
fig_sa_html = figs_html.get("DAMMAM (SA)")
fig_egy_html = figs_html.get("EL DEKHEILA (EGYPT)")
fig_kor_html = figs_html.get("INCHEON (KOREA)")


In [0]:
corn_hist_freight  = f"""
<!DOCTYPE html>
<html>
<head>
    <title>Corn Historical Freight rates</title>
    <style>
        body {{
            font-family: Arial, sans-serif;
            padding: 20px;
        }}
        h2 {{
            margin-top: 40px;
            color: #2c3e50;
        }}
        .chart-container {{
            margin-bottom: 50px;
        }}
    </style>
</head>
<body>
    <h1>Corn Historical Freight Rates</h1>
    <h2>Vietnam</h2>
    <div class="chart-container">{fig_viet_html}</div>
    
    <h2>Saudi Arabia</h2>
    <div class="chart-container">{fig_sa_html}</div>

    <h2>Egypt</h2>
    <div class="chart-container">{fig_egy_html}</div>

    <h2>Korea</h2>
    <div class="chart-container">{fig_kor_html}</div>

</body>
</html>
"""
corn_hist_freight_report_bytes = corn_hist_freight.encode("utf-8")

test_2=['florian.girardi-ext@ldc.com']
      
# Send email with embedded chart and table
LDCDataAccessLayerPy.mail.mail_send(
    to=grains,
    subject=f'CORN HISTORICAL FREIGHT RATES {start_date.strftime("%d-%m")}',
    from_addr="florian.girardi-ext@ldc.com",
    mime_type="html",
    body='Attached the report',
    attachment={"corn_historical_freight.html": corn_hist_freight_report_bytes,"hsitorical_freight_rates_corn.xlsx": excel_buffer_pivot_df_arg}
)
